# RAG — Retrieval Augmented Generation

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*¿Cómo le doy conocimiento propio a un LLM sin reentrenarlo?*

## Recap Clases 1-2 → conexión con hoy

```
  Clase 1                            Clase 2
  ───────                            ───────
  Texto → Tokens → Embeddings        Embeddings → Transformer → Texto generado
       BoW  TF-IDF  Vectores 384D        Attention   Prompting   LLM API
                │                              │
                └──────── HOY ─────────────────┘
                     RAG une retrieval con generación
```

| Lo que ya sabemos | Lo que resolvemos hoy |
|---|---|
| Embeddings capturan significado semántico | Los usamos para **recuperar** documentos relevantes |
| TF-IDF busca por coincidencia de palabras | Lo combinamos con búsqueda semántica |
| El LLM genera texto a partir de un prompt | Le **inyectamos contexto** antes de generar |
| Context window tiene límites | RAG selecciona **solo lo relevante** |


## Recorrido de la clase

| Bloque | Tema |
|---|---|
| **B1** | ¿Por qué RAG? + aplicaciones típicas + LLM vs RAG |
| **B2** | Pipeline paso a paso + chunking + vector stores + ejemplo Santa Fe |
| **B3** | Búsqueda híbrida (BM25 + semántica) + reranking |
| **B4** | RAG avanzado: HyDE, parent-child, Graph + Multi-Hop + troubleshooting |

Vamos a **construir el RAG paso a paso** con demos ejecutables y ejercicios.


## ⚙️ Setup

**Las prácticas son notebooks separados** que corren en Google Colab. Cada uno
se autoinstala las dependencias y configura las credenciales al inicio.

**Dependencias usadas a lo largo de la clase** (cada notebook hace su propio
`pip install`):

```bash
pip install sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib
```

**Pre-requisitos:**
- Cuenta gratuita en Groq → https://console.groq.com/keys
- Google account para Colab (gratis)

> El código que durante la clase muestre en pantalla va a estar en los notebooks
> linkeados al final de cada bloque. Las slides son sólo conceptuales.


# Bloque 1
## ¿Por qué RAG?

---

*El LLM sabe mucho, pero no sabe lo tuyo.*

## El problema del LLM puro

### Tres limitaciones que no se resuelven con mejor prompting

| Limitación | Ejemplo | Consecuencia |
|---|---|---|
| **Knowledge cutoff** | "¿Qué pasó en las elecciones de ayer?" | El modelo no tiene datos recientes |
| **Alucinaciones** | "Citame el artículo 47 del reglamento de la UTN" | Inventa contenido con total confianza |
| **Context window** | 50.000 documentos internos de una empresa | No caben ni en 1M de tokens |

```
  Opción 1: Re-entrenar el modelo     → costoso, lento, no práctico
  Opción 2: Meter todo en el prompt   → no escala, "Lost in the Middle"
  Opción 3: RAG                       → recuperar solo lo relevante ✓
```

> **RAG = no cambiás el modelo, cambiás lo que le das para leer.**

> 📖 *Lewis, P., et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. NeurIPS 2020. https://arxiv.org/abs/2005.11401*

## RAG — Retrieval Augmented Generation

**Idea central:** antes de que el LLM responda, buscar información relevante y ponerla en el prompt.

```
  ┌────────────────┐         ┌────────────────┐         ┌─────────────────────┐
  │  Base de       │ indexar │  Vector        │  top-k  │  Prompt =           │
  │  conocimiento  │ ──────▶ │  Store         │ ──────▶ │  chunks + query     │
  └────────────────┘         └───────┬────────┘         └──────────┬──────────┘
                                     │ buscar                      │
                             ┌───────┴────────┐                    ▼
                             │     Query      │              ┌──────────┐
                             └────────────────┘              │   LLM    │
                                                             └────┬─────┘
                                                                  ▼
                                                             Respuesta
```

**Los 4 pasos:**
1. **INDEXAR:** documentos → chunks → embeddings → vector store
2. **CONSULTAR:** query → embedding → buscar top-k chunks similares
3. **AUGMENTAR:** armar prompt = system + chunks recuperados + pregunta
4. **GENERAR:** el LLM responde usando los chunks como contexto

## LLM clásico vs LLM con RAG

| Capacidad | LLM clásico | LLM + RAG |
|---|---|---|
| **Conocimiento posterior al training** | ❌ Imposible | ✅ Vía retrieval sobre fuentes recientes |
| **Conocimiento privado / propietario** | ❌ Requiere fine-tuning | ✅ Solo indexar — el modelo queda intacto |
| **Citas y trazabilidad** | ❌ Inventa fuentes o no las da | ✅ Cada respuesta se rastrea a chunks específicos |
| **Reducción de alucinaciones** | ❌ Confianza alta en respuestas inventadas | ✅ Forzado a basarse en contexto (faithfulness) |
| **Razonamiento sobre documentos largos** | ⚠️ Limitado al context window | ✅ Solo trae lo relevante (evita lost-in-the-middle) |
| **Actualización del conocimiento** | ⚠️ Re-entrenar (caro y lento) | ✅ Re-indexar (rápido, barato, incremental) |
| **Costo por consulta** | ✅ Más barato | ⚠️ Mayor (retrieval + generation) |
| **Latencia** | ✅ Menor | ⚠️ Mayor (extra retrieval step) |

> Las dos últimas filas son los **tradeoffs reales** que RAG impone. Por todo lo demás, RAG es la opción default cuando hay corpus propio.


## Aplicaciones típicas de RAG (2025-2026)

- **Soporte / Customer success** — Q&A sobre docs, helpdesks internos, triage de tickets. Base que cambia frecuentemente; respuesta con citas.
- **Legal** — jurisprudencia, revisión de contratos, compliance. Citas obligatorias; vocabulario técnico; corpus enorme.
- **Salud / Medicina** — decisión clínica sobre papers y guías; resumen de historia clínica. Knowledge cutoff crítico; auditabilidad regulatoria.
- **Finanzas / Regulatorio** — Q&A sobre filings, due diligence, riesgo. Documentos largos y técnicos; rastreabilidad obligatoria.
- **Ingeniería / Código** — Q&A sobre codebase, runbooks, postmortems, generación de tests. Repos grandes, contexto distribuido.
- **Educación** — tutor sobre apuntes/libros, asistente de cursada, prep de examen. Cada estudiante consulta lo mismo desde ángulos distintos.
- **Investigación científica** — literature review, búsqueda semántica, hipótesis. Volumen masivo; terminología específica.
- **Enterprise / Knowledge ops** — "ask the company": wikis, políticas, HR. Datos privados que no entran al training de un LLM público.


## Cuándo conviene RAG (y cuándo no)

### Patrón común a todos los casos

**(1) corpus propio o nicho** + **(2) requerimiento de citas/groundedness** + **(3) actualización frecuente**.

### Cuándo RAG *no* es la respuesta

- Tareas **sin necesidad de conocimiento externo** — traducción, redacción creativa, conversación general.
- Datos que **caben en el context window** (< 200K tokens) → mandalos directo, sin retrieval.
- Tareas donde lo que falta es **razonamiento estructurado**, no información — cálculo numérico exacto, código complejo.


# Bloque 2
## Pipeline RAG naive

---

*De documentos sueltos a respuestas fundamentadas, paso a paso.*

## El pipeline RAG paso a paso

```
  FASE 1: INDEXACIÓN (se hace una vez)
  ─────────────────────────────────────────────────────────────────
  Documentos  ──▶  Chunking  ──▶  Embeddings  ──▶  Vector Store
                                                        │
                                                        │ (persiste)
  FASE 2: CONSULTA (cada pregunta)                      │
  ─────────────────────────────────────────────────────────────────
  Query  ──▶  Embedding  ──▶  Retrieval  ◀─────────────┘
                                  │
                                  ▼
                           Top-k chunks
                                  │
                                  ▼
                    Augmentation (prompt = chunks + query)
                                  │
                                  ▼
                                LLM  ──▶  Respuesta
```

Hoy implementamos cada paso de este diagrama.

## Chunking — partir documentos en fragmentos

### ¿Por qué? Un documento de 10 páginas no cabe como un solo embedding.

| Estrategia | Cómo funciona | Cuándo usarla |
|---|---|---|
| **Tamaño fijo** | Cada N caracteres, con overlap | Default simple, documentos homogéneos |
| **Por oración** | Partir en `. ` `? ` `! ` | Cuando cada oración es autocontenida |
| **Por párrafo** | Partir en `\n\n` | Documentos bien estructurados |
| **Recursivo** | Probar `\n\n` → `\n` → `. ` → espacio | LangChain default, el más robusto |

```
  Chunking fijo (200 chars, overlap 50):
  ┌──────────────┐
  │  Chunk 1     │
  │  chars 0-199 │
  └──────┬───────┘
         │overlap│
         ┌───────┴──────┐
         │  Chunk 2     │
         │  chars 150-349│
         └──────────────┘
```

**Overlap:** repetir N caracteres entre chunks para no cortar ideas a la mitad.  
Regla práctica: overlap = 10-20% del tamaño del chunk.

## Tamaño de chunk — el parámetro más importante

| Tamaño | Precisión | Contexto | Costo / Latencia | Cuándo usarlo |
|---|---|---|---|---|
| **64-128 tokens** | 🟢 Alta | 🔴 Bajo | 🟢 Bajo | Q&A factual corta, lookups exactos |
| **256-512 tokens** | 🟡 Media | 🟡 Medio | 🟡 Medio | **Default — el sweet spot** para la mayoría |
| **1024+ tokens** | 🔴 Baja | 🟢 Alto | 🔴 Alto | Documentos legales/técnicos con razonamiento largo |

### Reglas de bolsillo

```
  ┌────────────────────────────────────────────────────────────────────┐
  │ overlap     = 10-20% del tamaño del chunk                          │
  │ k           = 5-10 (cuántos chunks recuperás)                      │
  │ contexto    = k × chunk_size + prompt + query ≤ 50% del window     │
  └────────────────────────────────────────────────────────────────────┘
```


## Cómo ajustar el chunk size en la práctica

1. Empezá con **chunk=512, overlap=50, k=5** y un eval offline mínimo.
2. Si faltan **detalles** en la respuesta → **bajá chunk, subí k**.
3. Si la respuesta pierde **coherencia** entre chunks → **subí chunk, mantené k**.
4. Si **latencia o costo** duelen → **bajá k antes de bajar chunk**.

> No hay una respuesta universal. **Es un parámetro que se afina con eval offline.**


## Embeddings — el modelo que usamos en RAG

Recap rápido de Clase 1: cada chunk se transforma en un vector que captura su significado semántico.

Tres decisiones que pesan más en RAG que en Clase 1:

1. **Mismo modelo para chunks y queries.**
   Si embebés con modelos distintos, los vectores viven en espacios distintos y la similitud no significa nada.

2. **Modelo multilingüe vs específico.**
   Corpus en español → `paraphrase-multilingual-MiniLM-L12-v2` (~85MB) o `intfloat/multilingual-e5-large` (~2.2GB, más calidad, más lento).
   Corpus en inglés → el [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) tiene los rankings actualizados.

3. **Dimensionalidad y costo.**
   384D / 768D / 1024D — más dimensiones = más recall, más memoria, más latencia. Default razonable: 384D.

> En la práctica de hoy usamos `paraphrase-multilingual-MiniLM-L12-v2`: liviano, multilingüe, corre en Colab CPU sin GPU.


## Vector stores — BD para embeddings

### ¿Qué es un vector store?

Una base de datos especializada en guardar vectores y buscar los más similares.

```
  Base relacional (SQL):              Vector store:
  ─────────────────────               ─────────────
  SELECT * FROM docs                  "buscar los 3 chunks
  WHERE categoria = 'IA'              más parecidos a esta query"
  → coincidencia EXACTA               → similitud APROXIMADA

  Busca por: igualdad, rango          Busca por: distancia en espacio N-dim
  Índice: B-tree                      Índice: HNSW, IVF, PQ
```

### Características generales que comparten

- **Distancia** — cosine, dot product, euclidean (cosine es lo más común para embeddings).
- **Índice ANN** (Approximate Nearest Neighbor) — HNSW es el default; trade-off entre recall y latencia.
- **Metadata filtering** — además del vector, filtrar por campos estructurados (`autor='X'`, `fecha > Y`).
- **Persistencia** — in-memory para prototipos; disk/cloud para producción.
- **Escalabilidad** — millones a billones de vectores; sharding y replicación.


## Opciones de vector store

| Vector store | Modo | Cuándo elegirlo |
|---|---|---|
| **ChromaDB** | Open source, in-memory o local | Prototipos, clases, corpus chico — **el que usamos hoy** |
| **Pinecone** | SaaS cloud | Producción gestionada, sin DevOps |
| **Weaviate / Qdrant** | Open source, self-hosted | Producción con control total |
| **pgvector** | Extensión de PostgreSQL | Ya tenés Postgres y querés sumar vectores |
| **FAISS** | Librería (Meta) | Búsqueda batch, sin servidor |

> En esta clase usamos **ChromaDB** porque es in-memory, sin instalación, y nos deja enfocarnos en el pipeline RAG.


## Augmentation — el prompt que une todo

### El paso clave: construir un prompt que combine chunks recuperados con la pregunta.

```
  ┌─────────────────────────────────────────────────┐
  │  SYSTEM                                         │
  │  "Sos un asistente que responde basándose       │
  │   SOLO en el contexto proporcionado.            │
  │   Si no hay info suficiente, decilo."           │
  ├─────────────────────────────────────────────────┤
  │  USER                                           │
  │                                                 │
  │  Contexto:                                      │
  │  [1] (Arquitectura) "Los modelos de ML se..."   │
  │  [2] (MLOps) "MLOps aplica prácticas de..."     │
  │  [3] (MLOps) "Los contenedores Docker con..."   │
  │                                                 │
  │  Pregunta: ¿Cómo se despliegan modelos en prod? │
  ├─────────────────────────────────────────────────┤
  │  ASSISTANT                                      │
  │  "Según los documentos, los modelos se..."      │
  └─────────────────────────────────────────────────┘
```

**Reglas del prompt RAG:**
1. El system prompt **restringe** al LLM a usar solo el contexto dado
2. Los chunks se numeran para que el LLM pueda citar fuentes
3. La pregunta va **al final** (recency bias: el modelo presta más atención al final)

## Ejemplo end-to-end — multas de tránsito en Santa Fe

**Usuario:** *"¿Cuál es el plazo de prescripción para multas de tránsito en la Ciudad de Santa Fe?"*

```
  ┌─────────────────────────────────────────────────────────────────────┐
  │ 1. INDEXACIÓN (offline, se hace una vez)                             │
  │    • Código de Faltas + Resoluciones + Jurisprudencia local          │
  │    • Chunking semántico (~400 tokens, overlap 50)                    │
  │    • Embeddings multilingüe → ChromaDB                               │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 2. RETRIEVAL (online, por query)                                     │
  │    • Embed de la pregunta del usuario                                │
  │    • Top-5 chunks: Art. 47 (prescripción), Art. 52 (cómputo de plazo),│
  │      resumen de jurisprudencia 2023 sobre suspensión del plazo       │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 3. AUGMENTATION                                                      │
  │    • Prompt = system("citá fuentes") + chunks + query                │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 4. GENERACIÓN                                                        │
  │    • "Según el Art. 47 del Código de Faltas de Santa Fe,             │
  │       la prescripción es de 2 años desde la notificación.            │
  │       El plazo se suspende durante el proceso administrativo         │
  │       (Art. 52). Fuente: [Art. 47, Código de Faltas]"                │
  └─────────────────────────────────────────────────────────────────────┘
```


## Ejemplo end-to-end — multas de tránsito en Santa Fe

**Usuario:** *"¿Cuál es el plazo de prescripción para multas de tránsito en la Ciudad de Santa Fe?"*

### Por qué este caso es paradigmático para RAG

- **Datos públicos pero dispersos** — no entran en el training de ningún LLM global.
- **Auditabilidad obligatoria** — la respuesta debe rastrearse al artículo (no es opcional en lo legal).
- **Actualización constante** — los reglamentos cambian; re-entrenar el LLM no es viable.
- **Dominio localizado** — ningún modelo genérico lo cubre con la precisión necesaria.


## 🛠 Práctica del Bloque 2 — Pipeline RAG naive

📓 [`clase03/notebooks/rag/practica_completa.ipynb`](notebooks/rag/practica_completa.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_completa.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Corpus:** programa real de la **Cátedra IA UTN-FRSF 2026** (5 documentos: programa, condiciones de cursada, modalidad, evaluación, TPs).

**Lo que vas a hacer:**
- **Parte 1 — Motivación:** LLM puro vs RAG hardcoded (por qué hace falta retrieval).
- **Parte 2 — Pipeline naive a mano:** corpus → chunking → embeddings → ChromaDB → retrieval → augmentation → `rag_query()`.
- **Parte 3 — Probamos el pipeline** con tres queries representativas:
  1. **Lookup directo** — día, horario y aula (un sólo chunk).
  2. **Multi-fact** — criterios para aprobar la materia (varios chunks).
  3. **Edge case** — fecha del TP3 (no existe en el corpus → esperamos abstención, no alucinación).


# Bloque 3
## Hybrid Search

---

*A veces necesitás palabras exactas. A veces necesitás significado. Lo ideal: ambos.*

## BM25 — recap rápido

En Clase 1 vimos BoW y TF-IDF. **BM25** es la evolución de TF-IDF para búsqueda:

```
  BM25(query, doc) = Σ  IDF(término) × TF_saturada(término, doc) × norm_largo
                    término∈query
```

Mejoras sobre TF-IDF *(Term Frequency × Inverse Document Frequency)*:
- **Saturación de TF:** la 10ma aparición de una palabra no suma tanto como la 1ra.
- **Normalización por largo:** docs cortos no se penalizan injustamente.

### ¿Cuándo aporta sumar BM25 a la búsqueda semántica?

| Tipo de query | Tendencia |
|---|---|
| Vocabulario natural ("¿qué es un banco comercial?") | La búsqueda con embeddings suele alcanzar — captura el concepto aunque las palabras no coincidan. |
| Jerga técnica o siglas ("iliquidez", "HNSW", "ChromaDB") | BM25 puede rescatar la palabra exacta si el embedding la pasó por alto. |
| Conceptos amplios ("seguridad en IA") | Embeddings entienden la intención; BM25 sólo matchea literal. |

## Combinar BM25 + embeddings — weighted sum

Tenés dos retrievers que asignan scores en escalas distintas. ¿Cómo los mezclás?

**Opción 1 — Weighted sum (intuitiva pero frágil):**

`score(d) = α · emb_norm(d) + (1−α) · bm25_norm(d)`

Donde:
- **`d`** — un documento (chunk) del corpus al que le estás calculando el score.
- **`emb_norm(d)` / `bm25_norm(d)`** — score que da cada retriever, normalizado a `[0,1]`.
- **`α`** — peso entre 0 y 1 que controla cuánto pesa cada retriever:
  - `α = 1` → sólo embeddings
  - `α = 0` → sólo BM25
  - `α = 0.5` → mitad y mitad
  - típico: `α ∈ [0.5, 0.7]` (algo más a embeddings).

**Cuándo falla:** si BM25 y embeddings no se ponen de acuerdo sobre cuál chunk es el más relevante, el promedio le da score medio a todos y la respuesta correcta puede quedar fuera del top-k.

## Combinar BM25 + embeddings — RRF

**Opción 2 — Reciprocal Rank Fusion (RRF, estándar de industria):**

`score(d) = Σᵢ 1 / (k + rankᵢ(d))`   con k=60

Donde:
- **`d`** — un documento (chunk) del corpus.
- **`rankᵢ(d)`** — posición de `d` en el ranking del retriever `i` (BM25, embeddings, etc.). Si `d` quedó 1ro, `rankᵢ(d) = 1`.
- **`k`** — constante de amortiguación (60 es el default histórico). Suaviza el efecto de los rankings muy bajos.

Sumás recíprocos del **ranking**, no de scores. Si un chunk está en el top-3 de **cualquier** retriever, queda con score competitivo — sin importar la escala del otro. Es lo que usan Elasticsearch, OpenSearch, Vespa.

## Reranking — el segundo paso

Hasta acá todo el pipeline es **bi-encoder**: los embeddings de query y documentos se calculan **por separado** y se comparan después. Es rápido (indexás millones de docs offline), pero impreciso — la query nunca "ve" al documento.

Un **cross-encoder** procesa el par `(query, doc)` **juntos** por una red neuronal. Mucho más preciso, pero también mucho más lento: no se puede precalcular.

**Pipeline de 2 etapas** (lo mejor de los dos mundos):

```
  query
    │
    ▼
  [Bi-encoder]      BM25 + dense + RRF     →  top 6-10 candidatos rápidos
    │
    ▼
  [Cross-encoder]   reranking preciso      →  top 3 finales
```

Filtrás mucho con velocidad, elegís poco con precisión.

> **Modelos típicos:** `BAAI/bge-reranker-base` (multilingual, ~280 MB), `cross-encoder/ms-marco-MiniLM-L-6-v2` (inglés, ~80 MB).

# Bloque 4
## RAG avanzado

---

*El pipeline naive funciona. Estas técnicas lo hacen funcionar mejor.*

## HyDE — Hypothetical Document Embeddings

### Problema: la query del usuario es corta; los documentos son largos.

El embedding de "cómo monitorear un modelo" no se parece al embedding de un párrafo técnico sobre MLOps.

**Idea:** pedirle al LLM que **invente** un documento hipotético que responda la pregunta, y buscar con ESE embedding.

```
  Pipeline normal:     Query → embed(query) → buscar → chunks

  Pipeline HyDE:       Query → LLM genera doc hipotético
                                    ↓
                              embed(doc_hipotético) → buscar → chunks
```

**¿Por qué funciona?** El embedding del documento hipotético está más cerca del "espacio" de los documentos reales que el embedding de la query corta.

**Costo:** una llamada extra al LLM por cada query (latencia + tokens).

> 📖 *Gao, L., et al. (2023). Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE). ACL 2023. https://arxiv.org/abs/2212.10496*

## HyDE — ejemplo con multas de tránsito

**Query del usuario:** *"¿Cuál es el plazo de prescripción para multas de tránsito en la Ciudad de Santa Fe?"*

**Paso 1 — el LLM inventa una respuesta** (sin retrieval, sólo con lo que sabe):

*"El plazo de prescripción para multas de tránsito en la Ciudad de Santa Fe es de **2 años** contados desde la notificación de la infracción. Dicho plazo puede suspenderse durante la tramitación del proceso administrativo. Las multas no abonadas dentro de ese período prescriben automáticamente conforme al Código de Faltas provincial."*

**Paso 2 — buscamos con el embedding de ese texto** (no con el de la query original).

### Lo clave: no importa si lo que inventó es correcto

Tal vez el Código de Faltas real dice "1 año" y el LLM dijo "2 años". **No importa.** Lo que pesa es la **forma** del texto, no la verdad:

- La query corta vive en "espacio de preguntas" (interrogativa, casual).
- Los docs reales viven en "espacio de documentos legales" (declarativos, formales, con jerga).
- El doc inventado vive **también** en el espacio de documentos → matchea mejor contra el Art. 47 real que la query original.

## Parent-child chunks — contexto completo

### Problema: chunks chicos = retrieval preciso pero sin contexto. Chunks grandes = retrieval impreciso.

**Solución:** indexar chunks chicos (hijos) pero recuperar el chunk grande (padre).

```
  Documento original:
  ┌────────────────────────────────────────────────────┐
  │  Párrafo completo sobre seguridad en IA (padre)    │
  │  ┌──────────┐ ┌──────────┐ ┌──────────┐           │
  │  │ Oración 1│ │ Oración 2│ │ Oración 3│  (hijos)  │
  │  │ (indexar) │ │ (indexar) │ │ (indexar) │           │
  │  └──────────┘ └──────────┘ └──────────┘           │
  └────────────────────────────────────────────────────┘

  Query → match con Oración 2 (hijo)
       → devolver Párrafo completo (padre)
       → el LLM tiene más contexto para responder
```

**Implementación:** cada chunk hijo guarda un `parent_id` en metadata.
Al recuperar, se busca por hijo pero se devuelve el padre.

**Trade-off:**
- Más contexto para el LLM → mejor respuesta
- Más tokens por chunk → más costo, posible ruido


## Multi-Hop RAG — razonamiento sobre varios documentos

### Ejemplo sencillo

*"¿La empresa que fundó Steve Jobs tiene sede en la misma ciudad donde nació?"*

Para responder hace falta combinar **dos hechos** que están en documentos distintos:

1. **Hop 1** → recuperar: *"Steve Jobs fundó Apple"*
2. **Hop 2** → recuperar: *"Apple tiene sede en Cupertino, California"*
3. **Hop 3** → recuperar: *"Steve Jobs nació en San Francisco, California"*
4. **Síntesis** → misma California, distinta ciudad → **no**.

```
  Naive RAG:                       Multi-Hop RAG:
  ──────────                       ─────────────
  embed(query) → top-5             retrieve → analizar → re-query
  (los 5 chunks no contienen       cada paso responde a una sub-pregunta
   los 3 hechos juntos)            que el LLM formula a partir del anterior
```

**Cuándo usarlo:** preguntas que combinan info de ≥ 2 fuentes, comparaciones entre entidades, evolución temporal.

**Cuándo no:** Q&A de un solo concepto, lookups simples — naive ya alcanza.


## Graph RAG — knowledge graph como índice

### Ejemplo sencillo

> Corpus: 200 papers de investigación médica.
> Query: *"¿Qué tratamientos comparten autores con el Dr. García?"*

En lugar de buscar chunks por similitud, se construye un **grafo de entidades** y se navega por relaciones:

```
  Extracción (offline, con LLM):           Query (online):
  ─────────────────────────                ──────────────
  [Dr. García] ──autor de──▶ [Paper 1]     1. Buscar nodo [Dr. García]
  [Paper 1]    ──trata──────▶ [Diabetes]   2. Traverse autor → papers
  [Dr. López]  ──autor de──▶ [Paper 1]     3. Traverse coautores → otros papers
  [Dr. López]  ──autor de──▶ [Paper 7]     4. Traverse esos papers → tratamientos
  [Paper 7]    ──trata──────▶ [Hipertensión]
                                           Respuesta: Diabetes, Hipertensión
```

**Cuándo usarlo:** dominios **entidad-céntricos** (legal, biomedicina, investigación) donde las **relaciones** importan tanto como los textos.

**Cuándo no:** corpus principalmente narrativo o documental.

> 📖 *Edge, D., et al. (2024). From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* https://arxiv.org/abs/2404.16130


## Naive vs Hybrid vs Advanced RAG — recap

| Componente | **Naive** | **Hybrid** | **Advanced** |
|---|---|---|---|
| **Chunking** | Tamaño fijo, sin overlap | Semántico + overlap | Jerárquico (parent-child) |
| **Retrieval** | Solo dense (embeddings) | Dense + sparse (BM25) | Multi-hop, graph traversal |
| **Reranking** | Ninguno | Cross-encoder | Cross-encoder sobre multi-hop |
| **Query enrichment** | No | No | HyDE, query rewriting con LLM |
| **Costo / Latencia** | 🟢 Bajo | 🟡 Medio | 🔴 Alto |
| **Cuándo elegirlo** | Prototipo, corpus chico | Producto en serio | Dominios complejos (legal, médico) |


## Problemas comunes en RAG — troubleshooting (1/2)

| Síntoma observable | Causa probable | Mitigación |
|---|---|---|
| Top-k trae chunks **irrelevantes** | Embedding model mal elegido; distribución de queries ≠ corpus | Hybrid search (BM25 + dense), reranking con cross-encoder |
| Respuestas que **ignoran** los chunks | Prompt débil, alucinación del LLM | Faithfulness eval, prompt con "basate SOLO en…", structured output |
| **Recall bajo** — falta info que sí existe | Chunks muy chicos; embedding sub-óptimo; falta query rewriting | Parent-child chunks, HyDE, query expansion |
| **Latencia inaceptable** | Reranking sobre demasiados candidatos; embeddings no cacheados | Reranking sólo top-20; cache de queries frecuentes; embedding quantization |


## Problemas comunes en RAG — troubleshooting (2/2)

| Síntoma observable | Causa probable | Mitigación |
|---|---|---|
| **Costo alto** | Sin caching; embedding model caro; sobre-recuperación | Semantic cache; modelos OSS para embeddings; ajustar `k` |
| **Datos desactualizados** | Re-indexación manual; sin pipeline de ingesta | Pipeline incremental con CDC; versionar chunks; alerta de staleness |
| **Multi-documento falla** | Naive retrieval no enlaza chunks disjuntos | Multi-hop retrieval, graph RAG, agentic retrieval |
| **Vocabulario de dominio** mal entendido | Embedding genérico no captura jerga técnica | Fine-tuned embeddings (BGE, GTE); glosario de términos; query rewriting |

> Estos son **diagnósticos**, no recetas. Cada caso requiere medir si la mitigación realmente mueve la métrica.


## 🛠 Práctica — Naive vs Hybrid sobre un corpus legal real

📓 [`clase03/notebooks/rag/practica_legal.ipynb`](notebooks/rag/practica_legal.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/practica_legal.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Corpus:** Ley argentina **21.526 de Entidades Financieras** (67 artículos, ~91k chars) — un texto legal real, denso y con jerga técnica.

**Lo que vas a ver:**
- Construir el pipeline con **LangChain**: `Chroma` (dense) + `BM25Retriever` + `EnsembleRetriever` (RRF).
- **Caso 1** — query con vocabulario natural → **dense alcanza**.
- **Caso 2** — query con jerga técnica ("iliquidez") → **hybrid rescata** porque BM25 capta la palabra exacta.
- **Sección de exploración** — el cross-encoder reranker está armado para que pruebes sobre tus queries: puede mejorar precisión, puede filtrar señal útil. Medilo en tu caso.


## Lo que vimos hoy

| Bloque | Concepto clave |
|---|---|
| B1 — ¿Por qué RAG? | Knowledge cutoff, alucinaciones, context limits; aplicaciones reales (legal/salud/código/etc.) |
| B2 — Pipeline naive | Chunking → embeddings → vector store (ChromaDB) → retrieval → augmentation; ejemplo Santa Fe |
| B3 — Hybrid + Reranking | BM25 (léxico) + dense (vectores) combinados con RRF; cross-encoder como segunda pasada |
| B4 — RAG avanzado | HyDE, parent-child, Multi-Hop, Graph RAG, troubleshooting |

**Stack que usamos hoy:**
```
sentence-transformers · chromadb · rank_bm25 · langchain · cross-encoder · groq
```

**Notebooks de práctica:**
- `practica_completa.ipynb` — RAG naive de punta a punta sobre el programa de la cátedra.
- `practica_legal.ipynb` — naive vs hybrid con LangChain sobre la Ley 21.526.


---

## 📚 Bibliografía — RAG

### Libros de referencia
- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). Cap. 14 — Question Answering. https://web.stanford.edu/~jurafsky/slp3/

### Papers fundamentales
- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS. https://arxiv.org/abs/2005.11401
- Robertson, S. & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.*
- Cormack, G. V. et al. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual rank learning methods.* SIGIR. https://dl.acm.org/doi/10.1145/1571941.1572114
- Gao, L. et al. (2023). *HyDE — Precise Zero-Shot Dense Retrieval without Relevance Labels.* ACL. https://arxiv.org/abs/2212.10496
- Liu, N. F. et al. (2023). *Lost in the Middle — How Language Models Use Long Contexts.* TACL. https://arxiv.org/abs/2307.03172
- Edge, D. et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* https://arxiv.org/abs/2404.16130

### Documentación / herramientas
- **LangChain** — https://python.langchain.com/
- **ChromaDB** — https://docs.trychroma.com/
- **Sentence-Transformers** — https://www.sbert.net/
- **rank_bm25** — https://github.com/dorianbrown/rank_bm25
- **Cross-encoders (BGE reranker)** — https://huggingface.co/BAAI/bge-reranker-base


## ¿Qué viene?

Hoy construimos el RAG. Lo que sigue completa el ciclo:

---

### Evaluación y monitoreo de sistemas LLM
Cómo **medir y operar** sistemas LLM en producción — se aplica tanto a RAG como a cualquier sistema con LLMs.

- **Fundamentos** — por qué eval de LLMs es distinto, reference-based vs free, detección de alucinaciones, tracing.
- **Pre-deploy (eval offline)** — golden datasets, LLM-as-judge, RAGAS deep dive, safety + red-teaming, armar tu propio benchmark.
- **Deploy** — A/B, shadow, canary, prompt versioning, model registry, auto-rollback.
- **Producción** — logging, cost/latency, drift detection, feedback loops.
- **Práctica end-to-end con Arize AX** — un chatbot Q&A instrumentado sobre los temas de las clases 1-3.

### Agentes
Después: **agentes** — LLMs que deciden qué herramienta usar, planifican multi-step, y orquestan acciones (incluido el retrieval) de forma autónoma.
